# News Sentiment Data Preparation

## Data Source

Supplier-related news data were obtained through automated extraction from **Google News** using supplier names as search queries. The extraction process collected publicly available news articles mentioning suppliers included in the contract dataset.

The raw dataset contains article-level observations with the following core attributes:

- `supplier_number` — internal supplier identifier  
- `supplier` — supplier name  
- `title` — article headline  
- `source` — news publisher  
- `published` — publication timestamp  
- `link` — unique article URL  

To assess the relevance and sentiment of each article, a Large Language Model (LLM) was used to evaluate the content of each headline.


## LLM-Based Article Assessment

Each article was processed using a structured prompt to classify:

- **Relevance score** (scale 1–5)  
- **Sentiment** (positive, neutral, negative)  
- **Reasoning** (brief justification)

The purpose of this step was to distinguish meaningful supplier-related events from generic industry news and to generate interpretable risk signals for downstream modeling.

Only articles published from **2015 onwards** were considered for analysis. This temporal restriction ensures consistency with modern supplier monitoring practices and avoids outdated or sparse historical coverage.

RAW NEWS
   ↓
Extract year
   ↓
Remove duplicates
   ↓
Run sentiment
   ↓
Aggregate to supplier-year
   ↓
Create Join_Year
   ↓
Merge to contract-year

In [1]:
import pandas as pd
from pydantic import BaseModel, Field
from typing import Literal
from datetime import date

In [2]:
df_news_clean = pd.read_csv("/Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/Data/raw/news_cleaned_llm.csv")

In [3]:
df_news_clean.shape

(9313, 9)

In [4]:
df_news_clean.head(20)

,supplier_number,supplier,title,source,published,link,relevance_score,sentiment,reasoning
0,160089,LEFMEDPACKS,Explore Top 10 Packaging Industry Trends and I...,StartUs Insights,2026-01-16 08:00:00,https://news.google.com/rss/articles/CBMiggFBV...,2.0,neutral,Generic packaging industry trends article with...
1,161813,POLYPEPTIDE BELGIUM,Case study - Carrier Supplies Four Chillers To...,Carrier,2024-07-10 23:08:03,https://news.google.com/rss/articles/CBMiqgFBV...,3.5,positive,This case study specifically involves Polypept...
2,163204,AMCOR FLEXIBLES,Amcor Wins Silver for Earth Sense Recycled Fil...,Mirage News,2026-03-20 06:18:00,https://news.google.com/rss/articles/CBMiiwFBV...,4.5,positive,The article directly concerns Amcor Flexibles ...
3,163204,AMCOR FLEXIBLES,Amcor to ramp up North American flexibles prod...,Packaging Dive,2025-11-12 08:00:00,https://news.google.com/rss/articles/CBMirgFBV...,4.2,positive,The headline specifically references Amcor’s f...
4,163204,AMCOR FLEXIBLES,Amcor wins two WorldStar Global Packaging Awar...,Interplas Insights,2026-03-16 13:32:41,https://news.google.com/rss/articles/CBMi1gFBV...,4.0,positive,This recent (5 days ago) article clearly repor...
5,163204,AMCOR FLEXIBLES,Is Amcor (AMCR) Using Its New Bond Deal to Qui...,Sahm,2026-03-14 21:01:12,https://news.google.com/rss/articles/CBMizwFBV...,4.0,neutral,The headline clearly refers to Amcor (ticker A...
6,163204,AMCOR FLEXIBLES,Amcor Flexibles North America Plans Offering o...,TipRanks,2026-03-05 08:00:00,https://news.google.com/rss/articles/CBMixgFBV...,4.0,neutral,This headline is clearly about Amcor Flexibles...
7,163204,AMCOR FLEXIBLES,Amcor celebrates milestone for healthcare pack...,Packaging Scotland,2026-03-16 12:40:46,https://news.google.com/rss/articles/CBMioAFBV...,4.0,positive,The headline specifically references Amcor’s h...
8,163204,AMCOR FLEXIBLES,Amcor expects flat sales volumes to continue -...,Resource Recycling,2026-02-06 08:00:00,https://news.google.com/rss/articles/CBMinAFBV...,4.0,neutral,The article specifically discusses Amcor’s sal...
9,163204,AMCOR FLEXIBLES,Amcor boosts North American protein packaging ...,PlasticsToday,2025-11-12 08:00:00,https://news.google.com/rss/articles/CBMitwFBV...,4.0,positive,This article directly reports on Amcor (and it...


In [5]:
# Parse date and create year
df_news_clean = df_news_clean.copy()

df_news_clean["published"] = pd.to_datetime(
    df_news_clean["published"],
    errors="coerce"
)

df_news_clean["year"] = df_news_clean["published"].dt.year

In [6]:
df_news_clean["supplier_number"] = (
    df_news_clean["supplier_number"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

df_news_clean["year"] = df_news_clean["year"].astype("Int64")

In [7]:
# Remove duplicate articles
print("Duplicate links:", df_news_clean.duplicated("link").sum())

df_news_clean = df_news_clean.drop_duplicates(
    subset=["link"]
).copy()

Duplicate links: 1


In [8]:
# Check year range after extraction (>=2015)
print(
    "Year range:",
    df_news_clean["published"].min(),
    "to",
    df_news_clean["published"].max()
)

Year range: 2015-01-05 08:00:00 to 2026-03-20 11:31:44


In [9]:
# Keep only relevant enough articles
df_news_clean = df_news_clean[
    df_news_clean["relevance_score"] >= 3
].copy()

In [10]:
# Check distribution of articles per year
articles_per_year = (
    df_news_clean["year"]
    .value_counts()
    .sort_index()
)

print(articles_per_year)

year
2015      39
2016      45
2017      76
2018      81
2019     119
2020     145
2021     214
2022     267
2023     433
2024     575
2025    1411
2026    1510
Name: count, dtype: Int64


In [11]:
df_news_clean = df_news_clean[
    ~df_news_clean["title"]
    .str.contains(
        "obituary",
        case=False,
        na=False
    )
].copy()

In [12]:
df_news_clean = df_news_clean[
    df_news_clean["year"].notna()
].copy()

In [13]:
df_news_clean["Join_Year"] = (
    df_news_clean["year"] + 1
).astype("Int64")

In [14]:
sentiment_map = {
    "negative": -1,
    "neutral": 0,
    "positive": 1
}

df_news_clean["sentiment_score"] = (
    df_news_clean["sentiment"].map(sentiment_map)
)

In [15]:
df_news_yearly = (
    df_news_clean
    .groupby(["supplier_number", "Join_Year"], as_index=False)
    .agg(
        article_count=("link", "count"),
        negative_count=("sentiment", lambda x: (x == "negative").sum()),
        positive_count=("sentiment", lambda x: (x == "positive").sum()),
        neutral_count=("sentiment", lambda x: (x == "neutral").sum()),
        avg_sentiment_score=("sentiment_score", "mean"),
        avg_relevance_score=("relevance_score", "mean"),
        max_relevance_score=("relevance_score", "max")
    )
)

In [16]:
df_news_yearly = (
    df_news_clean
    .groupby(["supplier_number", "Join_Year"], as_index=False)
    .agg(
        article_count=("link", "count"),
        negative_count=("sentiment", lambda x: (x == "negative").sum()),
        positive_count=("sentiment", lambda x: (x == "positive").sum()),
        neutral_count=("sentiment", lambda x: (x == "neutral").sum()),
        avg_sentiment_score=("sentiment_score", "mean"),
        avg_relevance_score=("relevance_score", "mean"),
        max_relevance_score=("relevance_score", "max")
    )
)

In [17]:
df_news_yearly["negative_ratio"] = (
    df_news_yearly["negative_count"] / df_news_yearly["article_count"]
)

df_news_yearly["has_high_relevance_negative_news"] = (
    (df_news_yearly["negative_count"] > 0) &
    (df_news_yearly["max_relevance_score"] >= 4)
).astype(int)

In [18]:
df_news_clean.head(20)

,supplier_number,supplier,title,source,published,link,relevance_score,sentiment,reasoning,year,Join_Year,sentiment_score
1,161813,POLYPEPTIDE BELGIUM,Case study - Carrier Supplies Four Chillers To...,Carrier,2024-07-10 23:08:03,https://news.google.com/rss/articles/CBMiqgFBV...,3.5,positive,This case study specifically involves Polypept...,2024,2025,1
2,163204,AMCOR FLEXIBLES,Amcor Wins Silver for Earth Sense Recycled Fil...,Mirage News,2026-03-20 06:18:00,https://news.google.com/rss/articles/CBMiiwFBV...,4.5,positive,The article directly concerns Amcor Flexibles ...,2026,2027,1
3,163204,AMCOR FLEXIBLES,Amcor to ramp up North American flexibles prod...,Packaging Dive,2025-11-12 08:00:00,https://news.google.com/rss/articles/CBMirgFBV...,4.2,positive,The headline specifically references Amcor’s f...,2025,2026,1
4,163204,AMCOR FLEXIBLES,Amcor wins two WorldStar Global Packaging Awar...,Interplas Insights,2026-03-16 13:32:41,https://news.google.com/rss/articles/CBMi1gFBV...,4.0,positive,This recent (5 days ago) article clearly repor...,2026,2027,1
5,163204,AMCOR FLEXIBLES,Is Amcor (AMCR) Using Its New Bond Deal to Qui...,Sahm,2026-03-14 21:01:12,https://news.google.com/rss/articles/CBMizwFBV...,4.0,neutral,The headline clearly refers to Amcor (ticker A...,2026,2027,0
6,163204,AMCOR FLEXIBLES,Amcor Flexibles North America Plans Offering o...,TipRanks,2026-03-05 08:00:00,https://news.google.com/rss/articles/CBMixgFBV...,4.0,neutral,This headline is clearly about Amcor Flexibles...,2026,2027,0
7,163204,AMCOR FLEXIBLES,Amcor celebrates milestone for healthcare pack...,Packaging Scotland,2026-03-16 12:40:46,https://news.google.com/rss/articles/CBMioAFBV...,4.0,positive,The headline specifically references Amcor’s h...,2026,2027,1
8,163204,AMCOR FLEXIBLES,Amcor expects flat sales volumes to continue -...,Resource Recycling,2026-02-06 08:00:00,https://news.google.com/rss/articles/CBMinAFBV...,4.0,neutral,The article specifically discusses Amcor’s sal...,2026,2027,0
9,163204,AMCOR FLEXIBLES,Amcor boosts North American protein packaging ...,PlasticsToday,2025-11-12 08:00:00,https://news.google.com/rss/articles/CBMitwFBV...,4.0,positive,This article directly reports on Amcor (and it...,2025,2026,1
10,163204,AMCOR FLEXIBLES,Amcor invests in recyclable packaging facility...,Packaging Insights,2026-03-06 06:08:41,https://news.google.com/rss/articles/CBMijgFBV...,5.0,positive,"Direct, recent news of Amcor investing in a ne...",2026,2027,1


In [19]:
print(df_news_yearly.shape)

print(
    "Duplicate supplier-year rows:",
    df_news_yearly.duplicated(["supplier_number", "Join_Year"]).sum()
)

df_news_yearly.head()

(835, 11)
Duplicate supplier-year rows: 0


,supplier_number,Join_Year,article_count,negative_count,positive_count,neutral_count,avg_sentiment_score,avg_relevance_score,max_relevance_score,negative_ratio,has_high_relevance_negative_news
0,1011795,2017,1,0,1,0,1.000,3.0000,3.0,0.0,0
1,1011795,2019,2,0,2,0,1.000,3.5000,4.0,0.0,0
2,1011795,2020,3,0,3,0,1.000,3.5000,4.0,0.0,0
3,1011795,2022,8,0,7,1,0.875,3.1875,4.0,0.0,0
4,1011795,2023,8,0,7,1,0.875,3.0625,3.5,0.0,0


In [20]:
print(
    "Unique suppliers:",
    df_news_yearly["supplier_number"].nunique()
)

Unique suppliers: 219


In [21]:
df_news_yearly["negative_count"].describe() # sparse risk signal

count    835.000000
mean       0.737725
std        2.566049
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max       33.000000
Name: negative_count, dtype: float64

In [22]:
len(df_news_clean)

4913

In [23]:
from master_thesis.config import DATA_RAW

df_news_yearly.to_csv(DATA_RAW / "news.csv", index=False)
print("Saved to:", DATA_RAW / "news.csv")


Saved to: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/Data/raw/news.csv
